In [2]:
import requests
import json

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry


OBJECTIVE: 

Claws data from:

`
open-meteo.com/en/docs
`

In [3]:
# Settings for Singapore

LATITUDE = 1.3521
LONGITUDE = 103.8198
TIMEZONE = "Asia/Singapore"



In [ ]:
# Taken from: https://open-meteo.com/en/docs


# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": LATITUDE,
	"longitude": LONGITUDE,
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)



Coordinates: 1.3708258867263794°N 103.8023681640625°E
Elevation: 46.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-08-13 00:00:00+00:00       26.799999
1   2026-08-13 01:00:00+00:00       27.950001
2   2026-08-13 02:00:00+00:00       29.200001
3   2026-08-13 03:00:00+00:00       30.400000
4   2026-08-13 04:00:00+00:00       30.900000
..                        ...             ...
163 2026-08-19 19:00:00+00:00       25.600000
164 2026-08-19 20:00:00+00:00       25.600000
165 2026-08-19 21:00:00+00:00       25.700001
166 2026-08-19 22:00:00+00:00       25.799999
167 2026-08-19 23:00:00+00:00       26.000000

[168 rows x 2 columns]


In [11]:
hourly_dataframe

,date,temperature_2m
0,2026-08-13 00:00:00+00:00,26.799999
1,2026-08-13 01:00:00+00:00,27.950001
2,2026-08-13 02:00:00+00:00,29.200001
3,2026-08-13 03:00:00+00:00,30.400000
4,2026-08-13 04:00:00+00:00,30.900000
...,...,...
163,2026-08-19 19:00:00+00:00,25.600000
164,2026-08-19 20:00:00+00:00,25.600000
165,2026-08-19 21:00:00+00:00,25.700001
166,2026-08-19 22:00:00+00:00,25.799999


In [8]:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&hourly=temperature_2m"
)
response

<Response [200]>

In [10]:
response.json()

{'latitude': 52.52,
 'longitude': 13.419998,
 'generationtime_ms': 0.07164478302001953,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 38.0,
 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C'},
 'hourly': {'time': ['2026-08-13T00:00',
   '2026-08-13T01:00',
   '2026-08-13T02:00',
   '2026-08-13T03:00',
   '2026-08-13T04:00',
   '2026-08-13T05:00',
   '2026-08-13T06:00',
   '2026-08-13T07:00',
   '2026-08-13T08:00',
   '2026-08-13T09:00',
   '2026-08-13T10:00',
   '2026-08-13T11:00',
   '2026-08-13T12:00',
   '2026-08-13T13:00',
   '2026-08-13T14:00',
   '2026-08-13T15:00',
   '2026-08-13T16:00',
   '2026-08-13T17:00',
   '2026-08-13T18:00',
   '2026-08-13T19:00',
   '2026-08-13T20:00',
   '2026-08-13T21:00',
   '2026-08-13T22:00',
   '2026-08-13T23:00',
   '2026-08-14T00:00',
   '2026-08-14T01:00',
   '2026-08-14T02:00',
   '2026-08-14T03:00',
   '2026-08-14T04:00',
   '2026-08-14T05:00',
   '2026-08-14T06:00',
   '2026-08-14T07:00